In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

trip_types = ["intercity_one_way", "intercity_round_trip",
              "airport_pickup", "airport_drop", "local_package"]
trip_type = np.random.choice(trip_types, n, p=[0.30, 0.20, 0.12, 0.13, 0.25])

value_range = {
    "intercity_one_way": (3000, 12000),
    "intercity_round_trip": (6000, 25000),
    "airport_pickup": (800, 2500),
    "airport_drop": (800, 2500),
    "local_package": (1500, 4500),
}
cancel_prob = {
    "intercity_one_way": 0.16,
    "intercity_round_trip": 0.12,
    "airport_pickup": 0.08,
    "airport_drop": 0.10,
    "local_package": 0.12,
}

booking_value = [np.random.randint(*value_range[t]) for t in trip_type]
status = [np.random.choice(["served", "cancelled"], p=[1 - cancel_prob[t], cancel_prob[t]])
          for t in trip_type]

df = pd.DataFrame({
    "booking_id": range(1, n + 1),
    "trip_type": trip_type,
    "booking_value": booking_value,
    "status": status,
})

cancellation_rate = (df["status"] == "cancelled").mean() * 100
revenue_lost = df.loc[df["status"] == "cancelled", "booking_value"].sum()

print("Cancellation rate:", round(cancellation_rate, 1), "%")
print("Revenue lost: Rs", revenue_lost)

Cancellation rate: 13.0 %
Revenue lost: Rs 465367


In [5]:
cancelled = df[df["status"] == "cancelled"].copy()
n_c = len(cancelled)

cancelled["cancel_reason"] = np.random.choice(
    ["price_issue", "no_travel_assistance", "itinerary_time_change", "cab_segment_change"],
    n_c, p=[0.35, 0.20, 0.28, 0.17])
cancelled["contacted_support"] = np.random.choice(["yes", "no"], n_c, p=[0.29, 0.71])
cancelled["rebooked"] = np.random.choice(["yes", "no"], n_c, p=[0.29, 0.71])

self_cancel_rate = (cancelled["contacted_support"] == "no").mean() * 100
rebooking_rate = (cancelled["rebooked"] == "yes").mean() * 100

print(cancelled["cancel_reason"].value_counts())
print("Self-cancellation rate:", round(self_cancel_rate, 1), "%")
print("Rebooking rate:", round(rebooking_rate, 1), "%")

cancel_reason
price_issue              26
itinerary_time_change    16
no_travel_assistance     15
cab_segment_change        8
Name: count, dtype: int64
Self-cancellation rate: 83.1 %
Rebooking rate: 29.2 %


In [6]:
revenue_recovered = cancelled.loc[cancelled["rebooked"] == "yes", "booking_value"].sum()
print("Revenue recovered: Rs", revenue_recovered)

by_reason = cancelled.groupby("cancel_reason")["rebooked"].apply(
    lambda x: (x == "yes").mean() * 100).round(1)
print(by_reason)

Revenue recovered: Rs 107246
cancel_reason
cab_segment_change       37.5
itinerary_time_change    31.2
no_travel_assistance     26.7
price_issue              26.9
Name: rebooked, dtype: float64


In [7]:
df["lost"] = np.where(df["status"] == "cancelled", df["booking_value"], 0)

summary = df.groupby("trip_type").agg(
    bookings=("booking_id", "count"),
    cancelled=("status", lambda x: (x == "cancelled").sum()),
    revenue_lost=("lost", "sum"),
)
summary["cancellation_rate_%"] = (summary["cancelled"] / summary["bookings"] * 100).round(1)
print(summary)

                      bookings  cancelled  revenue_lost  cancellation_rate_%
trip_type                                                                   
airport_drop                67          6          8777                  9.0
airport_pickup              61          5          7689                  8.2
intercity_one_way          154         23        179637                 14.9
intercity_round_trip        87         13        211847                 14.9
local_package              131         18         57417                 13.7
